<a href="https://colab.research.google.com/github/nosadchiy/public/blob/main/NonStationary_Inventory_K.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Compute order up to levels for time-dependent stochastic demand. Fixed ordering cost K. Zero lead time.

In [25]:
import numpy as np
from scipy.stats import poisson


def expected_cost_given_post_inventory(b: float, h: float, lam: float, opt_cost_to_go: np.ndarray) -> np.ndarray:
    """
    Vectorized expected (single-period holding+shortage + continuation) cost
    as a function of post-decision inventory y = 0..N.

    Returns G[y] for y=0..N.
    """
    T = int(len(opt_cost_to_go))          # T = N+1
    demand = np.arange(T)                # 0..N (truncation support)

    P = poisson.pmf(demand, lam)
    P = P / P.sum()                      # renormalization

    y = np.arange(T)[:, None]            # (T,1)
    d = demand[None, :]                  # (1,T)

    inv_end = np.maximum(y - d, 0).astype(int)           # (T,T) in [0..N]
    holding = h * inv_end
    backorder = b * np.maximum(d - y, 0)
    continuation = opt_cost_to_go[inv_end]

    per_scenario_cost = holding + backorder + continuation  # (T,T)
    G = per_scenario_cost @ P                                # (T,)

    return G.astype(float)


def suffix_argmin_inclusive(a: np.ndarray):
    """
    For each x, returns:
      best_idx_ge[x] = argmin_{i >= x} a[i]
      best_val_ge[x] = min_{i >= x} a[i]
    """
    n = len(a)
    best_idx = np.empty(n, dtype=int)
    best_val = np.empty(n, dtype=float)

    best_idx[-1] = n - 1
    best_val[-1] = a[-1]
    for i in range(n - 2, -1, -1):
        if a[i] <= best_val[i + 1]:
            best_idx[i] = i
            best_val[i] = a[i]
        else:
            best_idx[i] = best_idx[i + 1]
            best_val[i] = best_val[i + 1]
    return best_idx, best_val


def suffix_argmin_strict(a: np.ndarray):
    """
    For each x, returns:
      best_idx_gt[x] = argmin_{i > x} a[i]   (strictly greater)
      best_val_gt[x] = min_{i > x} a[i]
    If no i>x exists (x==N), sets idx=N and val=+inf.
    """
    n = len(a)
    best_idx_gt = np.empty(n, dtype=int)
    best_val_gt = np.empty(n, dtype=float)

    best_idx_ge, best_val_ge = suffix_argmin_inclusive(a)

    # For x in 0..n-2, min over i>x is min over i>=x+1
    best_idx_gt[:-1] = best_idx_ge[1:]
    best_val_gt[:-1] = best_val_ge[1:]

    # For x==n-1, no strictly greater index
    best_idx_gt[-1] = n - 1
    best_val_gt[-1] = np.inf

    return best_idx_gt, best_val_gt


def solve_sS_policy(lambda_vec, b=9.0, h=1.0, K=50.0):
    """
    Backward induction for finite horizon with Poisson demand (truncated),
    holding cost h, shortage cost b, and fixed ordering cost K.

    Returns:
      s_t (length n): reorder thresholds
      S_t (length n): order-up-to levels
      order_flag[x,t]: whether to order at state x in period t
      y_star[x,t]: optimal post-decision inventory level (either x or S_t)
      cost_opt[x,t]: optimal cost-to-go
    """
    lambda_vec = np.asarray(lambda_vec, dtype=float)
    n = len(lambda_vec)

    N = int(4 * np.max(lambda_vec))   # same truncation rule as before

    cost_opt = np.zeros((N + 1, n + 1), dtype=float)   # terminal column = 0
    order_flag = np.zeros((N + 1, n), dtype=bool)
    y_star = np.zeros((N + 1, n), dtype=int)

    s = np.zeros(n, dtype=int)
    S = np.zeros(n, dtype=int)

    for t in range(n - 1, -1, -1):
        lam = float(lambda_vec[t])
        opt_cost_to_go = cost_opt[:, t + 1]

        # G[y] = expected cost if we start the period with post-decision inventory y
        G = expected_cost_given_post_inventory(b, h, lam, opt_cost_to_go)  # length N+1

        # Candidate "order-to" level (if we do order) is the minimizer of G over y
        # (ties broken to smallest y due to argmin)
        S_t = int(np.argmin(G))

        # For each state x:
        #   no order cost = G[x]
        #   order cost    = K + min_{y > x} G[y]
        best_y_gt, best_G_gt = suffix_argmin_strict(G)

        no_order_cost = G
        order_cost = K + best_G_gt

        do_order = order_cost < no_order_cost   # strict improvement => order
        # (if you want weak inequality, change to <=)

        # Optimal post-inventory: if order, go to argmin_{y>x} G[y]; else stay at x
        y_opt = np.arange(N + 1)
        y_opt[do_order] = best_y_gt[do_order]

        # Store DP results
        order_flag[:, t] = do_order
        y_star[:, t] = y_opt
        cost_opt[:, t] = np.where(do_order, order_cost, no_order_cost)

        # Derive (s_t, S_t) summary:
        # With fixed K and no per-unit order cost, optimal "order-to" is constant
        # across all x where ordering happens, and equals S_t (global minimizer of G).
        #
        # Threshold s_t = max x such that ordering is optimal.
        # Note: if never order, set s_t = -1 conventionally.
        if np.any(do_order):
            s_t = int(np.max(np.where(do_order)[0]))
        else:
            s_t = -1

        # If ordering happens, order-to should equal S_t; with truncation + ties,
        # there can be corner cases where best_y_gt[x] != S_t for some x>S_t.
        # The (s,S) structure applies on the region x<=s_t, so we report S_t.
        s[t] = s_t
        S[t] = S_t

        # Optionally, enforce exact (s,S) action map:
        # order to S_t only if x <= s_t; else don't order.
        if s_t >= 0:
            x = np.arange(N + 1)
            do_order_ss = x <= s_t
            order_flag[:, t] = do_order_ss
            y_star[:, t] = np.where(do_order_ss, S_t, x)
            # corresponding costs
            cost_opt[:, t] = np.where(do_order_ss, K + G[S_t], G)

    print("[s,S Policy] (s_t, S_t) for each period t=1..n:")
    for t in range(n):
        print(f"t={t+1}: s={s[t]}, S={S[t]}")

    return s, S, order_flag, y_star, cost_opt


if __name__ == "__main__":
    lambda_vec = [26, 8, 4, 2, 1] #vector of average demands per period of time (Poisson distributed)
    s, S, order_flag, y_star, cost_opt = solve_sS_policy(lambda_vec, b=9.0, h=1.0, K=50.0)

[s,S Policy] (s_t, S_t) for each period t=1..n:
t=1: s=24, S=41
t=2: s=6, S=16
t=3: s=-1, S=8
t=4: s=-1, S=4
t=5: s=-1, S=2
